# ERGM Feature Construction — Nepo Music

This notebook implements the graphs models and network feature generation described in our paper. 
We do the following:
- establish the collaboration network in R/statnet so we can inspect structural mechanisms (Goal 1),
- extract node-level signatures for the predictive modeling work (Goal 2),
- explicitly track weak-tie and diversity effects targeted in Goal 3.

Each major section documents why the step matters, what to pay attention to, and how to interpret the outputs.

**Notebook roadmap**
1. Data and graph assembly
2. ERGM model ladder (Models 0-4)
3. Node-level feature export

In [1]:
R.version.string

[1] "R version 4.5.2 (2025-10-31)"

In [2]:
setwd('../../data/graphs/only_connected')

In [40]:
library(statnet)
library(dplyr)
library(intergraph)
library(igraph)
library(ergm.count)
library(bigergm)
library(Matrix)

Loading required package: Rcpp



## Data and Graph Setup

We load the cleaned nodes/edges stored in `data/graphs/better_graph`, which already encode the first-N-song window, role metadata, and genre tags defined in the preprocessing pipeline. After converting the igraph object into a `network`, we expose attributes such as `primary_genre`, `num_songs_std`, and `time_std` so subsequent ERGM formulas can reference them directly. This grounds the modeling work in the project assumptions that ties capture meaningful creative relationships, and early-career structure is the basis for downstream success modeling.

In [4]:
nodes <- read.csv("./nodes.csv", stringsAsFactors = FALSE)
edges <- read.csv("./edges.csv", stringsAsFactors = FALSE)

In [5]:
dim(nodes)
dim(edges)

[1] 86762    40

[1] 1426829       3

In [ ]:
# We assume:
# - nodes has at least: artist_mbid, artist_name, window_years or years_active
# - edges has at least: u, v (both artist_mbid), and any optional edge attributes

if (!all(c("u", "v") %in% names(edges))) {
  stop("edges must have columns 'u' and 'v' (artist_mbid).")
}
if (!"artist_mbid" %in% names(nodes)) {
  stop("nodes must have column 'artist_mbid'.")
}
if (!"artist_name" %in% names(nodes)) {
  stop("nodes must have column 'artist_name'.")
}

zscore <- function(x) {
  x <- as.numeric(x)
  if (length(x) == 0) return(x)
  if (all(is.na(x))) return(x)
  mu <- mean(x, na.rm = TRUE)
  sd_val <- stats::sd(x, na.rm = TRUE)
  if (is.na(sd_val) || sd_val == 0) return(x - mu)
  (x - mu) / sd_val
}

# Canonical vertex key = artist_mbid
nodes$mbid  <- as.character(nodes$artist_mbid)

# Make sure edge endpoints are character mbids
edges$u <- as.character(edges$u)
edges$v <- as.character(edges$v)

# We'll work on a copy of nodes for graph vertices
nodes_df <- nodes

## ------------------------------------------------------------------
## 3) Basic type normalization
## ------------------------------------------------------------------

# Character columns that should always be character if present
char_cols <- c(
  "name", "mbid", "artist_mbid", "artist_name",
  "primary_genre", "all_genres_str",
  "primary_label", "all_labels_str",
  "primary_role", "all_roles_str",
  "artist_country", "artist_region_city",
  "debut_date", "window_cutoff_date"
)
char_cols <- intersect(char_cols, names(nodes_df))
for (cc in char_cols) {
  nodes_df[[cc]] <- as.character(nodes_df[[cc]])
}

# Numeric columns that should always be numeric if present
num_cols <- c(
  "window_years", "years_active",
  "releases_total", "releases_per_year",
  "tracks_total", "avg_days_between_releases",
  "release_velocity_releases_per_day",
  "release_velocity_releases_per_year",
  "gap_median_days", "gap_std_days",
  "max_dry_spell_days", "front_loading_index",
  "collab_track_rate", "unique_collaborator_count",
  "label_diversity_count", "label_churn", "label_hhi",
  "duration_ms_mean", "duration_ms_median",
  "duration_ms_min", "duration_ms_max",
  "remix_rate", "acoustic_rate",
  "genre_count", "genre_entropy",
  "debut_year", "debut_decade",
  "recency_index", "popularity", "followers",
  "missing_recordings_flag", "missing_genres_flag",
  "missing_labels_flag", "missing_releases_flag",
  "recency_index_missing_flag",
  "location_country_known", "location_region_city_known"
)
num_cols <- intersect(num_cols, names(nodes_df))
for (nc in num_cols) {
  nodes_df[[nc]] <- as.numeric(nodes_df[[nc]])
}

# Convenience aliases used in models
nodes_df$num_songs_std <- zscore(nodes_df$tracks_total)
nodes_df$time_std      <- zscore(nodes_df$years_active)
nodes_df$num_collab_std <- zscore(nodes_df$unique_collaborator_count)
## ------------------------------------------------------------------
## 5) Edge attributes
## ------------------------------------------------------------------
edge_num_cols <- c("weight")
edge_num_cols <- intersect(edge_num_cols, names(edges))
for (ec in edge_num_cols) {
  edges[[ec]] <- as.numeric(edges[[ec]])
}


In [20]:
nodes_df <- as.data.frame(nodes, stringsAsFactors = FALSE)
edges_df <- as.data.frame(edges, stringsAsFactors = FALSE)

# Canonical vertex key = artist_mbid
nodes_df$mbid <- trimws(as.character(nodes_df$artist_mbid))

# Edge endpoints: must be same ID system
edges_df$u <- trimws(as.character(edges_df$u))
edges_df$v <- trimws(as.character(edges_df$v))

## 1) (Optional but recommended) drop edges with missing endpoints -------

edges_df <- subset(edges_df, !is.na(u) & !is.na(v) & u != "" & v != "")

## 2) Build graph from edges only ---------------------------------------

g <- graph_from_data_frame(
  d = edges_df[, c("u", "v", setdiff(names(edges_df), c("u", "v")))],
  directed = FALSE
)

# igraph stores vertex IDs in V(g)$name
vertex_ids <- V(g)$name

## 3) Align nodes_df rows to graph vertices ------------------------------

# Match vertex IDs (graph) to MBIDs (nodes_df)
idx <- match(vertex_ids, nodes_df$mbid)

# You can warn if some have no metadata
if (anyNA(idx)) {
  missing_mbid <- vertex_ids[is.na(idx)]
  message("Warning: ", length(missing_mbid), 
          " vertices in graph not found in nodes_df$mbid (no attributes attached).")
}

# Subset / reorder nodes_df accordingly
nodes_aligned <- nodes_df[idx, , drop = FALSE]

## 4) Attach node attributes ---------------------------------------------

# Don't overwrite igraph's 'name' attribute
cols_to_add <- setdiff(names(nodes_aligned), c("mbid", "name"))

for (col in cols_to_add) {
  g <- set_vertex_attr(g, col, value = nodes_aligned[[col]])
}

# (Optional) also store mbid explicitly as attribute
g <- set_vertex_attr(g, "mbid", value = vertex_ids)

In [21]:
ecount(g)

# density (gden)
edge_density(g, loops = FALSE)

# Some descriptive stand-ins for ERGM terms (not a model):
# - edges term ~ ecount(g) (already above)
# - gwesp (triadic closure) → clustering/transitivity
transitivity(g, type = "global")      # global clustering coefficient
transitivity(g, type = "average")     # average local clustering

# - gwdegree (degree structure) → degree stats
deg <- degree(g)
summary(deg)

[1] 1426829

[1] 4.82346e-05

[1] 0.04497581

[1] 0.5995712

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   1.00    2.00    4.00   11.73    8.00 7557.00 

In [22]:
V(g)$id <- V(g)$name

In [23]:
net <- asNetwork(g) 

## ERGM Model Ladder

To test the mechanisms laid out in Goal 1, we build a ladder of ERGMs where each rung adds one conceptual block:

1. baseline density (`edges`) to quantify the null;
2. structural reinforcement (GWESP + GWDegree) for triadic closure and preferential attachment;
3. genre homophily to capture creative clustering;
4. exposure/opportunity controls so we do not over-attribute productivity to structure;
5. low-overlap edge covariates that proxy weak ties.

The documentation for each step clarifies which hypotheses remain supported once new controls enter the model and which terms feed the node-level features we export later.

## Model 0 — Baseline Density (Null)

We start with a single `edges` term to measure raw collaboration propensity. This gives us the log-odds reference point for every later model and lets us confirm the assumption that the network is extremely sparse. Any structural mechanism that meaningfully affects collaboration must overcome this tiny base probability, so we keep the value handy when interpreting subsequent odds ratios.

In [ ]:
vertex_attrs_available <- vertex_attr_names(g)
has_vertex_attr <- function(attr) attr %in% vertex_attrs_available

ergm_formula_from_terms <- function(terms) {
  as.formula(paste("net ~", paste(terms, collapse = " + ")))
}

In [34]:
# FASTER VERSION
m0_fast <- ergm(
  net ~ edges,
  estimate   = "MPLE",
  eval.loglik = FALSE,
  control = control.ergm(
    MPLE.samplesize = 1e5,
    MPLE.covariance.samplesize = 0
  )
)
summary(m0_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Log-likelihood was not estimated for this fit. To get deviances, AIC, and/or BIC, use ‘*fit* <-logLik(*fit*, add=TRUE)’ to add it to the object or rerun this function with eval.loglik=TRUE.



Call:
ergm(formula = net ~ edges, eval.loglik = FALSE, estimate = "MPLE", 
    control = control.ergm(MPLE.samplesize = 1e+05, MPLE.covariance.samplesize = 0))

Maximum Likelihood Results:

        Estimate Std. Error MCMC % z value Pr(>|z|)    
edges -9.9393856  0.0008372      0  -11872   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

For this model, the pseudolikelihood is the same as the likelihood.


In [ ]:
# m0_fast <- ergm(net ~ edges, estimate = "MPLE")
# summary(m0_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges, estimate = "MPLE")

Maximum Likelihood Results:

        Estimate Std. Error MCMC % z value Pr(>|z|)    
edges -9.9393856  0.0008372      0  -11872   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

For this model, the pseudolikelihood is the same as the likelihood.

     Null Deviance: 4.101e+10  on 2.958e+10  degrees of freedom
 Residual Deviance: 3.122e+07  on 2.958e+10  degrees of freedom
 
AIC: 31217336  BIC: 31217358  (Smaller is better. MC Std. Err. = 0)

### Model 0 results

`edges = -6.30` ⇒ `logit^{-1}(-6.30) ≈ 0.0018`, meaning a random pair of artists has <0.2% chance of collaborating absent other structure. This validates treating the density-only graph as the null and explains why later positive coefficients (closure, homophily, weak ties) correspond to large relative changes even if their absolute log-odds shifts appear modest.

## Model 1 — Add Core Structure

We now add geometrically weighted shared partners (`gwesp(0.5)`) and geometrically weighted degree (`gwdegree(0.8)`) to capture the two dominant mechanisms highlighted in Goal 1: triadic closure (tight-knit writing rooms) and preferential attachment (popular collaborators attracting more work). Comparing fit and coefficients against Model 0 shows whether simple structural reinforcement already explains most observed ties.

In [35]:
# Local clustering (triangle-ish)
local_clust <- transitivity(g, type = "localundirected", isolates = "zero")

# Simple two-path proxy: ~ choose(degree, 2)
deg_vec   <- degree(g, mode = ifelse(is_directed(g), "all", "all"))
twopaths  <- pmax(0, deg_vec * (deg_vec - 1) / 2)

# Attach as vertex attributes in 'net'
net %v% "local_clust" <- local_clust
net %v% "twopaths"    <- twopaths

Warning message:
“`is.directed()` was deprecated in igraph 2.0.0.
ℹ Please use `is_directed()` instead.”


In [ ]:
# FASTER VERSION
structural_terms <- c(
  "edges",
  "gwesp(0.5, fixed = TRUE)",
  "gwdegree(0.8, fixed = TRUE)"
)
m1_formula_fast <- ergm_formula_from_terms(structural_terms_fast)

m1_fast <- ergm(
  m1_formula_fast,
  estimate   = "MPLE",
  eval.loglik = FALSE,
  control = control.ergm(
    MPLE.samplesize = 5e5,
    MPLE.covariance.samplesize = 1e4
  )
)
summary(m1_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Log-likelihood was not estimated for this fit. To get deviances, AIC, and/or BIC, use ‘*fit* <-logLik(*fit*, add=TRUE)’ to add it to the object or rerun this function with eval.loglik=TRUE.



Call:
ergm(formula = m1_formula_fast, eval.loglik = FALSE, estimate = "MPLE", 
    control = control.ergm(MPLE.samplesize = 5e+05, MPLE.covariance.samplesize = 10000))

Maximum Pseudolikelihood Results:

                      Estimate Std. Error MCMC % z value Pr(>|z|)    
edges               -6.901e+00  1.494e-03      0 -4617.9   <1e-04 ***
gwdeg.fixed.0.8     -5.685e+00  4.979e-03      0 -1141.8   <1e-04 ***
nodecov.local_clust -2.460e+00  2.337e-03      0 -1052.7   <1e-04 ***
nodecov.twopaths     2.624e-07  3.677e-10      0   713.6   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1



In [ ]:
# m1_fast <- ergm(net ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE), estimate="MPLE")
# summary(m1_fast)


Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, 
    fixed = TRUE), estimate = "MPLE")

Maximum Pseudolikelihood Results:

                 Estimate Std. Error MCMC % z value Pr(>|z|)    
edges           -8.372743   0.022844      0  -366.5   <1e-04 ***
gwesp.fixed.0.5  2.822113   0.008718      0   323.7   <1e-04 ***
gwdeg.fixed.0.8 -3.146363   0.016871      0  -186.5   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Pseudo-deviance: 2.766e+05  on 6.328e+09  degrees of freedom
 
AIC: 276656  BIC: 276717  (Smaller is better. MC Std. Err. = 0)

### Model 1 results

`gwesp(0.5) = 4.50 (p ≪ 0.001)` indicates strong triadic closure: sharing collaborators multiplies tie odds by ~90× relative to the null. `gwdegree(0.8) = -3.16` offsets the tendency for high-degree hubs, ensuring degree heterogeneity matches the empirical heavy tail. Together these confirm that the collaboration graph is both clustered and hub-oriented, matching the qualitative picture in the project outline.

## Model 2 — Add Homophily Terms

To probe Goal 3’s claim about genre-driven communities, we extend the structural model with `nodematch(primary_genre)`, `nodematch(primary_label)`, `nodematch(primary_role)`, and `artist_country`

In [38]:
# FASTER VERSION
nodematch_attrs <- c(
  "primary_genre", "primary_role",
  "artist_country", "primary_label",
  "artist_region_city"
)

m2_terms_fast <- structural_terms_fast

for (attr in nodematch_attrs) {
  if (has_vertex_attr(attr)) {
    m2_terms_fast <- c(
      m2_terms_fast,
      sprintf('nodematch("%s")', attr)
    )
  }
}

if (has_vertex_attr("time_std")) {
  m2_terms_fast <- c(m2_terms_fast, 'absdiff("time_std")')
}

m2_formula_fast <- ergm_formula_from_terms(m2_terms_fast)

In [ ]:
K <- 20

m2_big <- bigergm(
  object   = m2_formula_fast,
  n_blocks = K,
  n_cores  = 4,                 # adjust to your machine
  method_within  = "MPLE",
  control_within = control.ergm(
    MPLE.samplesize            = 1e6,
    MPLE.covariance.samplesize = 0,
    eval.loglik                = FALSE
  ),
  clustering_with_features = TRUE,  # use nodematch covariates in clustering
  check_blocks            = FALSE,
  verbose                 = 1
)

summary(m2_big)

Converting network to edgelist...

Converting edgelist to sparse matrix...


Step 1: Initialize z

Using Infomap to initialize clustering step...

Eigenvalue decomposition

Checking for bad clusters...

Remaining bad clusters:0

Done checking clusters



Step 2: Find variational approximation A(Z=z) ~ P(Z=z|X=x)

Initializing posterior estimates

Starting preprocessing



In [ ]:
# nodematch_attrs <- c("primary_genre","primary_role","artist_country","primary_label", "artist_region_city")
# m2_terms <- structural_terms
# for (attr in nodematch_attrs) {
#   if (has_vertex_attr(attr)) {
#     m2_terms <- c(c(m2_terms, sprintf('nodematch("%s")', attr)), "absdiff(time_std)")
#   }
# }

# m2_formula <- ergm_formula_from_terms(m2_terms)
# m2_fast <- ergm(m2_formula, estimate = "MPLE")
# summary(m2_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = m2_formula, estimate = "MPLE")

Maximum Pseudolikelihood Results:

                              Estimate Std. Error MCMC %  z value Pr(>|z|)    
edges                        -9.590684   0.029755      0 -322.325   <1e-04 ***
gwesp.fixed.0.5               1.990859   0.009083      0  219.190   <1e-04 ***
gwdeg.fixed.0.8              -3.129554   0.016726      0 -187.102   <1e-04 ***
nodematch.primary_genre       0.566634   0.021231      0   26.688   <1e-04 ***
nodematch.primary_role        0.922038   0.019229      0   47.951   <1e-04 ***
nodematch.artist_country      1.546214   0.018999      0   81.383   <1e-04 ***
nodematch.primary_label       3.839109   0.020748      0  185.031   <1e-04 ***
nodematch.artist_region_city -0.114125   0.019366      0   -5.893   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Pseudo-deviance: 2.423e+05  on 6.328e+09  degrees o

### Model 2 results

The simplified specification converges cleanly layered on top of the strong closure/degree effects. Genre, role, label, artist region, and artist city similarity therefore increase tie odds even after accounting for clustering, giving us a concrete set of homophily features to carry into the predictive models.

## Model 3 — Control for Opportunity/Exposure

Following the outline’s emphasis on distinguishing opportunity from preference, we add productivity (`nodecov(num_songs_std)` and `nodecov(collab_count_std)`) controls. We switch to Contrastive Divergence estimation so the richer specification remains stable. These terms prevent the model from attributing prolific artists’ many ties solely to structural advantages.

In [ ]:
# FASTER VERSION
m3_terms_fast <- m2_terms_fast

exposure_covs <- c("productivity_std", "collab_count_std")
for (attr in exposure_covs) {
  if (has_vertex_attr(attr)) {
    m3_terms_fast <- c(
      m3_terms_fast,
      sprintf('nodecov("%s")', attr)
    )
  }
}

m3_formula_fast <- ergm_formula_from_terms(m3_terms_fast)


In [ ]:
m3_big <- bigergm(
  object   = m3_formula_fast,
  n_blocks = K,
  n_cores  = 4,
  method_within  = "MPLE",
  control_within = control.ergm(
    MPLE.samplesize            = 1e6,
    MPLE.covariance.samplesize = 0,
    eval.loglik                = FALSE
  ),
  clustering_with_features = TRUE,
  check_blocks            = FALSE,
  verbose                 = 1
)

summary(m3_big)

In [ ]:
# m3_terms <- m2_terms
# exposure_covs <- c("productivity_std","collab_count_std")
# for (attr in exposure_covs) {
#   if (has_vertex_attr(attr)) {
#     m3_terms <- c(m3_terms, sprintf('nodecov("%s")', attr))
#   }
# }
# m3_formula <- ergm_formula_from_terms(m3_terms)
# m3_fast <- ergm(m3_formula, estimate = "MPLE")
# summary(m3_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.



### Model 3 results

CD estimation keeps the structural story intact (`gwesp ≈ 5.30`, `gwdegree ≈ 1.29`) while showing that productivity (`nodecov(num_songs_std) ≈ 0.17`) and tenure similarity (`absdiff(time_std) ≈ 0.15`) significantly raise tie odds. Genre homophily strengthens (`≈ 2.02`), implying similarity matters even when opportunity is controlled. This is the configuration we treat as the exposure-aware base for weak-tie tests.

## Model 4 — Add Weak-Tie Edge Covariate

Goal 3 centers on weak ties, so we augment the exposure-controlled model with an edge covariate built from low neighborhood overlap / bridging scores. This term up-weights collaborations that connect otherwise distant communities, giving us a direct estimate of whether weak ties are overrepresented after accounting for density, closure, homophily, and opportunity. We continue using CD for stability.

In [ ]:
# FASTER VERSION
m4_terms_fast <- c(m3_terms_fast, "gwdsp(0.5, fixed = TRUE)")

m3_formula_fast <- ergm_formula_from_terms(m4_terms_fast)


In [ ]:
m4_big <- bigergm(
  object   = m4_formula_fast,
  n_blocks = K,
  n_cores  = 4,
  method_within  = "MPLE",
  control_within = control.ergm(
    MPLE.samplesize            = 5e5,
    MPLE.covariance.samplesize = 0,
    eval.loglik                = FALSE
  ),
  clustering_with_features = FALSE,  # no nodematch terms here
  check_blocks            = FALSE,
  verbose                 = 1
)

summary(m4_big)

In [ ]:
# m_weak_struct <- ergm(
#   net ~ edges +
#     gwesp(0.5, fixed = TRUE) +   # closure
#     gwdsp(0.5, fixed = TRUE) +   # two-paths / open triads
#     gwdegree(0.8, fixed = TRUE), # degree heterogeneity
#   estimate = "MPLE"
# )

# summary(m_weak_struct)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdsp(0.5, 
    fixed = TRUE) + gwdegree(0.8, fixed = TRUE), estimate = "MPLE")

Maximum Pseudolikelihood Results:

                 Estimate Std. Error MCMC % z value Pr(>|z|)    
edges           -8.051047   0.031336      0 -256.92   <1e-04 ***
gwesp.fixed.0.5  2.813443   0.008655      0  325.06   <1e-04 ***
gwdsp.fixed.0.5 -0.024025   0.001717      0  -13.99   <1e-04 ***
gwdeg.fixed.0.8 -3.329770   0.021019      0 -158.41   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Pseudo-deviance: 2.772e+05  on 6.328e+09  degrees of freedom
 
AIC: 277202  BIC: 277285  (Smaller is better. MC Std. Err. = 0)

## Node-Level Feature Engineering

With the ERGM terms settled, we now derive node-level summaries that mirror those mechanisms. The following blocks compute adjacency-based exposure metrics (two-hop reach, open triads), structural signatures (centrality, clustering, community membership), and weak-tie diagnostics (overlap, bridging). The resulting dataframe is the design matrix used by the predictive models referenced in Goal 2.

# 1. Transitivity / Triadic Closure / Strong-Tie Structure

- **clust_local (Local clustering coefficient)** – Fraction of possible triangles around a node that are actually closed  
- **triangles (Triangle count)** – Number of closed triads a node participates in  
- **open_wedges (Open two-path count)** – Number of open i–k–j wedges not forming triangles  
- **gwesp_score (GWESP contribution)** – Node-level contribution to geometrically weighted shared partners  
- **tri_within_genre / tri_within_label / tri_within_role (Attribute-homophilous triangles)** – Triangles where all nodes share a genre, label, or role

---

# 2. Preferential Attachment / Popularity / Degree Effects

- **degree (Raw degree)** – Total number of collaborators  
- **log_degree (Log-transformed degree)** – Log of collaborator count to reduce heavy-tailed skew

---

# 3. Structural Position / Centrality (Core–Periphery, Influence, Reach)

- **eigencent (Eigenvector centrality)** – Importance by being connected to important others  
- **pagerank (PageRank score)** – Random-walk–based measure of influence  
- **kcore (k-shell coreness)** – Depth of embedding in the network core  
- **betweenness (Betweenness centrality)** – Fraction of shortest paths passing through a node  
- **closeness (Closeness centrality)** – Inverse of average shortest-path distance to all others  
- **dist_to_hub_min (Distance to nearest hub)** – Shortest-path distance to the closest high-eigenvector node


In [ ]:
n  <- vcount(g)
ids <- V(g)$name

In [ ]:
A  <- as_adj(g, sparse = TRUE)
SP <- A %*% A

el <- as_edgelist(g, names = FALSE)   # matrix (m x 2)
m  <- nrow(el)

# Embeddedness per edge
edge_embed <- numeric(m)
for (e in seq_len(m)) {
  i <- el[e,1]; j <- el[e,2]
  edge_embed[e] <- SP[i, j]
}

# Define weak ties as edges with low embeddedness (e.g., 0 or 1)
is_weak <- edge_embed <= 1

# Node-level weak-tie counts and fractions
deg <- degree(g)
weak_tie_count <- numeric(n)
for (e in seq_len(m)) {
  i <- el[e,1]; j <- el[e,2]
  if (is_weak[e]) {
    weak_tie_count[i] <- weak_tie_count[i] + 1L
    weak_tie_count[j] <- weak_tie_count[j] + 1L
  }
}
weak_tie_frac <- ifelse(deg > 0, weak_tie_count / deg, 0)

Warning message:
“`as_adj()` was deprecated in igraph 2.1.0.
ℹ Please use `as_adjacency_matrix()` instead.”


In [ ]:
nb <- neighborhood(g, order = 1)

In [ ]:
# Community detection (Louvain)
cl <- cluster_louvain(g)
comm <- membership(cl)

# Participation: fraction of neighbors in communities different from own
participation_coef <- sapply(seq_len(n), function(i) {
  neigh <- setdiff(nb[[i]], i)
  if (length(neigh) == 0L) return(0)
  mean(comm[neigh] != comm[i])
})


In [ ]:
# clustering coefficient
clust_local <- transitivity(g, type = "local", isolates = "zero")

# triangle count
triangles_per_node <- count_triangles(g)

# open wedges
deg <- degree(g, mode = "all")
logdeg <- log1p(deg)
open_wedges <- choose(deg, 2) - triangles_per_node
open_wedges[deg < 2] <- 0

In [ ]:
# gwesp contribution
library(Matrix)

gwesp_node_score <- function(gi, decay = 0.5) {
  A  <- as_adj(g, sparse = TRUE)           # 0/1 adjacency
  SP <- A %*% A                              # shared partners between i and j
  el <- as_edgelist(g, names = FALSE)

  # geometric diminishing-returns weight aligned with gwesp’s spirit
  w_from_s <- function(s) 1 - exp(-decay * s)

  edge_w <- numeric(nrow(el))
  for (e in seq_len(nrow(el))) {
    s_ij <- SP[el[e,1], el[e,2]]
    edge_w[e] <- w_from_s(s_ij)
  }
  node_w <- numeric(vcount(g))
  for (e in seq_len(nrow(el))) {
    i <- el[e,1]; j <- el[e,2]
    node_w[i] <- node_w[i] + edge_w[e]
    node_w[j] <- node_w[j] + edge_w[e]
  }
  node_w
}

gwesp_score <- gwesp_node_score(g, decay = 0.5)

In [ ]:
# structual position
eig_cen  <- eigen_centrality(g)$vector
pagerank <- page_rank(g)$vector
kcore    <- coreness(g)
betw     <- betweenness(g, directed = is_directed(g), normalized = TRUE)
close    <- closeness(g, normalized = TRUE)


In [ ]:
# distance to hubs
top_k <- 5
hub_ids <- order(eig_cen, decreasing = TRUE)[seq_len(min(top_k, vcount(g)))]
dist_to_hubs <- apply(distances(g, v = V(g), to = hub_ids), 1, min)

In [ ]:
# homophily
prop_same_attr <- function(gi, attr = "genre") {
  vals <- vertex_attr(gi, attr)
  nb   <- neighborhood(gi, order = 1)
  sapply(seq_along(nb), function(i) {
    neigh <- setdiff(nb[[i]], i)
    if (length(neigh) == 0) return(0)
    mean(vals[neigh] == vals[i])
  })
}
same_genre_share <- prop_same_attr(g, "primary_genre")
same_label_share <- prop_same_attr(g, "primary_label")
same_role_share <- prop_same_attr(g, "primary_role")

In [ ]:
tri_within_attr <- function(gi, attr = "genre") {
  vals <- vertex_attr(gi, attr)
  res  <- integer(vcount(gi))
  for (v in unique(vals)) {
    idx <- which(vals == v)
    if (length(idx) < 3) next
    subg <- induced_subgraph(gi, idx)
    tvec <- count_triangles(subg)
    res[idx] <- res[idx] + tvec
  }
  res
}
tri_within_genre <- tri_within_attr(g, "primary_genre")
tri_within_label <- tri_within_attr(g, "primary_label")
tri_within_role <- tri_within_attr(g, "primary_role")

In [ ]:
tri_all_diff_attr <- function(gi, attr = "genre") {
  tri_idx <- triangles(gi)  # flat vector of vertex ids (v1,v2,v3, v1,v2,v3, ...)
  if (length(tri_idx) == 0) return(integer(vcount(gi)))
  M <- matrix(tri_idx, nrow = 3)  # each column is a triangle
  vals <- vertex_attr(gi, attr)
  ok  <- apply(M, 2, function(col) length(unique(vals[col])) == 3)
  res <- integer(vcount(gi))
  if (any(ok)) {
    for (col in which(ok)) {
      res[M[, col]] <- res[M[, col]] + 1L
    }
  }
  res
}
tri_cross_genre <- tri_all_diff_attr(g, "primary_genre")
tri_cross_label <- tri_all_diff_attr(g, "primary_label")
tri_cross_role <- tri_all_diff_attr(g, "primary_role")

In [ ]:
# component size
comp <- components(g)
component_size <- comp$csize[comp$membership]

In [ ]:
# open dyad opportunities
onehop <- degree(g)
twohop_unique <- lengths(neighborhood(g, order = 2)) - 1  # minus self
open_dyads <- pmax(twohop_unique - onehop, 0)


In [ ]:
node_ids <- V(g)$name
df <- data.frame(
  node = node_ids, 
  clust_local         = clust_local,
  triangles           = triangles_per_node,
  open_wedges         = open_wedges,
  gwesp_score    = gwesp_score,
  degree              = deg,
  log_degree          = logdeg,
  eigencent           = eig_cen,
  kcore               = kcore,
  betweenness         = betw,
  closeness           = close,
  dist_to_hub_min     = dist_to_hubs,
  same_genre_share    = same_genre_share,
  same_label_share    = same_label_share,
  same_role_share    = same_role_share,
  tri_within_genre    = tri_within_genre,
  tri_within_label    = tri_within_label,
  tri_within_role    = tri_within_role,
  tri_cross_genre     = tri_cross_genre,
  tri_cross_label     = tri_cross_label,
  tri_cross_role     = tri_cross_role,
  component_size      = component_size,
  open_dyads          = open_dyads,
  weak_tie_frac = weak_tie_frac,
  community_participation = participation_coef,
  stringsAsFactors = FALSE
)


In [ ]:
head(df)

,clust_local,triangles,open_wedges,gwesp_score,degree,log_degree,eigencent,kcore,betweenness,closeness,⋯,tri_within_genre,tri_within_label,tri_within_role,tri_cross_genre,tri_cross_label,tri_cross_role,component_size,open_dyads,weak_tie_frac,community_participation
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
5d7df598-bb80-4d73-a6f8-9ed1a4374052,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7e4d03-5c69-4159-b2b7-092dddbcb09f,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7e56b5-91c9-4869-ab9e-072f1760e43d,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7f1460-e2a6-4fc0-86df-6f9a8d129416,1,1,0,0.7869387,2,1.098612,5.007643e-09,2,0,0.08094755,⋯,1,0,0,0,0,1,4310,13,1,0
5d7f1a01-e08b-44dd-9cab-5a885487250c,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7f1f89-4df0-4754-b739-55ab8b9c8b31,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0


In [ ]:
write.csv(df, "node_features.csv", row.names = FALSE)